In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import math

import pandas as pd


import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots

In [3]:
import sys
import os
from pathlib import Path


def _uar(p):
    """Resolve a Unifying_Algo_Results path. Results shared with the main figure
    live in Main/; only supplemental-specific extras live in Supplemental/. If a
    Supplemental/ path is requested but the file lives in Main/, redirect to Main/."""
    p = str(p)
    if '/Supplemental/' in p:
        _main = p.replace('/Supplemental/', '/Main/')
        if os.path.exists(_main):
            return _main
    return p


In [4]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

# Helper Functions

In [5]:
#from splitflow.unifying_algo_analysis_helper import *
from splitflow.set_zorder_functions import *

Using device: cpu


In [6]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    #set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
    #set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)
    ax.set_rasterization_zorder(0)

    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xlim(-2, 2)
    ax.set_xticks([-200, -100, 0, 100, 200])

    if show_labels:
        ax.set_xticklabels(['$-2$', '$-1$', '$0$', '$1$', '$2$'])
    else:
        ax.set_xticklabels([])
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Aligned time (s)', labelpad=1)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [7]:
def mean_angle(angle):
    
    mean = np.arctan2( np.nanmean(np.sin(angle)), np.nanmean(np.cos(angle)) )
    return mean

def angle_distance(angle1, angle2):
    """
    Calculate the minimum distance between two angles.

    Parameters:
    -----------
    angle1, angle2 : float or array-like
        Angles in radians

    Returns:
    --------
    float or array
        Minimum distance between angles in radians 
        Range: [-π, π]
    """
    diff = angle1 - angle2
    # Wrap to [-π, π]
    distance = np.arctan2(np.sin(diff), np.cos(diff))
    return distance

In [8]:
FIGURE_NAME = 'supplemental_unifying_analysis_flies.svg'

In [9]:
TRANSLATION = True

In [10]:
COURSE_MARKER_SIZE = 2
COURSE_ALPHA_MULTIPLIER = 3

In [11]:
class LabelToMetadata:
    def __init__(self):
        self.flash = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/laminar_data_flash_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/unsteady_top_on_data_flash_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/stillair_data_flash_translation' + str(TRANSLATION) + '.parquet': [3, '#084a72ff', 'stillair'], 
                               }
        
        self.sham = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/laminar_data_sham_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/unsteady_top_on_data_sham_AUGMENTED_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/stillair_data_sham_translation' + str(TRANSLATION) + '.parquet': [3, '#084a72ff', 'stillair'], 
                               }
        
        self.WT_flash = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/WT_laminar_data_flash_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                                    #str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/unsteady_top_on_data_WT_flash_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'],
                                    str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/experiencedunsteady_top_on_data_WT_flash_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/WT_stillair_data_flash_translation' + str(TRANSLATION) + '.parquet': [3, '#084a72ff', 'stillair'], 
                               }
        
        self.WT_sham = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/WT_laminar_data_sham_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                                   #str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/unsteady_top_on_data_WT_sham_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'],
                                   str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/experiencedunsteady_top_on_data_WT_sham_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/WT_stillair_data_sham_translation' + str(TRANSLATION) + '.parquet': [3, '#084a72ff', 'stillair'], 
                               }
        
        self.CFD_casting = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/cfd_sim_steady_casting_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/cfd_sim_unsteady_casting_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'], 
                                 'None': [3, '#084a72ff', 'stillair'], 
                               }
        
        self.CFD_circling = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/cfd_sim_steady_circling_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/cfd_sim_unsteady_circling_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/cfd_sim_still_air_circle_translation' + str(TRANSLATION) + '.parquet': [3, '#084a72ff', 'stillair'], 
                               }
        
        self.unifying = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/new_unifying_laminar_translation' + str(TRANSLATION) + '.parquet': [1, '#991128ff', 'laminar'], 
                         str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/new_unifying_unsteady_translation' + str(TRANSLATION) + '.parquet': [2, '#2b75b3ff', 'unsteady'], 
                         str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/new_unifying_stillair_translation' + str(TRANSLATION) + '.parquet': [3, '#084a72ff', 'stillair'], 
                       }

In [12]:
def get_filename_for_wind_type(metadata, windtype):
    filename = None
    for key, val in metadata.items():
        if windtype in val:
            filename = key
    return filename

In [13]:
def get_trajec_filename_from_unifying_filename(unifying_filename):
    if 'cfd_sim' in unifying_filename:
        return str(Path('../../../Data/Simulated_Trajectory_Data')) + '/cfd_sim_circling_casting_all_wind_conditions_preprocessed.parquet'
    elif 'new_unifying' in unifying_filename:
        return str(Path('../../../Data/Simulated_Trajectory_Data')) + '/new_unifying.parquet'
    elif 'WT_laminar' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_laminar_wt_preprocessed_optotrigger_trimmed.hdf'
    elif 'unsteady_top_on_data_WT' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_splitflow_topon_wt_preprocessed_optotrigger_trimmed.parquet'
    elif 'all_top_on_data_WT' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_splitflow_topon_wt_preprocessed_optotrigger_trimmed.parquet'
    elif 'WT_stillair' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_stillair_wt_preprocessed_optotrigger_trimmed.hdf'
    elif 'laminar_data' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_laminar_c1xwt_preprocessed_optotrigger_trimmed.hdf'
    elif 'stillair_data' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_stillair_c1xwt_preprocessed_optotrigger_trimmed.hdf'
    elif 'unsteady' in unifying_filename and 'flash' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed.hdf'
    elif 'unsteady' in unifying_filename and 'sham' in unifying_filename:
        return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed_SHAM_AUGMENTED_merged.parquet'

In [14]:
def get_filenames_for_metadata_windtype(metadata, windtype):
    unifying_filename = get_filename_for_wind_type(metadata, windtype)
    trajectory_filename = None
    df = None
    unifying_algo_data = None
    
    if unifying_filename != 'None':
        unifying_algo_data = pd.read_parquet(_uar(unifying_filename))
    
        trajectory_filename = get_trajec_filename_from_unifying_filename(unifying_filename)
        if '.hdf' in trajectory_filename:
            df = pd.read_hdf(trajectory_filename)
        else:
            df = pd.read_parquet(trajectory_filename)
    
    else:
        unifying_algo_data = None

    print(unifying_filename)
    print(trajectory_filename)
    return unifying_algo_data, df

# Set condition

* 'flash'
* 'sham'
* 'WT_flash'
* 'WT_sham'

In [15]:
label = 'WT_flash'
label_to_metadata = LabelToMetadata()
fifi_label_course = label + '_' + 'course'
metadata = label_to_metadata.__getattribute__(label)

In [16]:
metadata

{'../../../Data/Unifying_Algo_Results/Supplemental/WT_laminar_data_flash_translationTrue.parquet': [1,
  '#991128ff',
  'laminar'],
 '../../../Data/Unifying_Algo_Results/Supplemental/experiencedunsteady_top_on_data_WT_flash_translationTrue.parquet': [2,
  '#2b75b3ff',
  'unsteady'],
 '../../../Data/Unifying_Algo_Results/Supplemental/WT_stillair_data_flash_translationTrue.parquet': [3,
  '#084a72ff',
  'stillair']}

In [17]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

# With real laminar data

In [18]:
show_labels = True

In [19]:
windtype = 'laminar'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_label_course, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 675 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1

    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems[label+'_'+windtype+'_n']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/WT_laminar_data_flash_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_laminar_wt_preprocessed_optotrigger_trimmed.hdf


# With real unsteady data

In [20]:
show_labels = False

In [21]:
windtype = 'unsteady'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)

if unifying_algo_data is not None:
    ax = layout.axes[(fifi_label_course, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 675 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1
    
    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )
    clean_labels(ax, show_labels)
    

    item = layout.svgitems[label+'_'+windtype+'_n']
    N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
    item.text = 'n=' + str(N_trajecs)

if unifying_algo_data is None:
    ax = layout.axes[(fifi_label_course, windtype)]
    fifi.mpl_functions.adjust_spines(ax, [])

../../../Data/Unifying_Algo_Results/Supplemental/experiencedunsteady_top_on_data_WT_flash_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_splitflow_topon_wt_preprocessed_optotrigger_trimmed.parquet


# With real stillair data

In [22]:
show_labels = False

In [23]:
windtype = 'stillair'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_label_course, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 675 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1
    
    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )
    clean_labels(ax, show_labels)



    item = layout.svgitems[label+'_'+windtype+'_n']
    N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
    item.text = 'n=' + str(N_trajecs)



../../../Data/Unifying_Algo_Results/Supplemental/WT_stillair_data_flash_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_stillair_wt_preprocessed_optotrigger_trimmed.hdf


In [24]:
layout.apply_svg_attrs()

layout.append_figure_to_layer(layout.figures[fifi_label_course], fifi_label_course, cleartarget=True)
layout.write_svg(FIGURE_NAME)

In [25]:
#data = diagnose_axis_elements(ax)